# DINO hazard ensemble: 128 predictions per image

This notebook performs GPU feature extraction, trains **128 independent bootstrap linear heads**,
and exports **128 raw p(hazard) values for each Pittsburgh image**. Ranking, uncertainty summaries,
lambda selection, metrics and plots belong in a separate analysis workflow. The only training log
is the optimization loss already computed for backpropagation; validation/test labels never guide
training or checkpoint selection. All 128 final heads are retained.

Use a Colab **T4 GPU**, including through the VS Code Colab extension. No Google Drive mount or
Google authorization is used. Supply the dataset archive through the anonymous file link or
local-upload option below, and provide an `HF_TOKEN` with DINOv3 access when prompted.

Outputs live at `/content/dino_hazard_bootstrap_runs/<run-id>/`. The final cell packages the run
as `/content/downloads/<run-id>.zip`. Download it before resetting/deleting the Colab runtime.
The ZIP includes all weights, raw predictions, bootstrap membership and provenance.

Start a **fresh 128-head run**. A 10-head run ZIP uses the previous implementation and is not a
resume checkpoint for this version. Completed heads within this version can be resumed.


## Exact configuration and its source

We reuse the validation-selected **hazard** configuration from `dino_sidewalk_data_3.ipynb`.
The selected configuration ID is **`68caf0783493ae93`**.

| Setting | Value used here |
|---|---|
| Training data | `sidewalk-data-3/pitt-train`: 1,276 records (527 hazard, 749 non-hazard) |
| Other-city data | The 22,912 `other-cities-train` records are not used for this Pitt-only configuration |
| Backbone | Frozen `facebook/dinov3-vitb16-pretrain-lvd1689m` (DINOv3 ViT-B/16) |
| Backbone revision | `5931719e67bbdb9737e363e781fb0c67687896bc` |
| Representation | 768-dimensional CLS vector; one linear logit plus sigmoid per head |
| Optimizer | Adam; learning rate **0.00003**, weight decay **0.01**, betas (0.9, 0.999), epsilon 1e-8 |
| Training duration | **5,000 optimizer updates per head**, batch size **64**; final checkpoint |
| Loss | Unweighted hazard binary cross-entropy with logits |
| Preprocessing | Dataset RGB 224×224 PNGs; full-image bicubic resize during dataset creation; ImageNet normalization |
| Precision | FP16 frozen-backbone computation, FP32 cached vectors and head optimization |
| Bootstrap members | **128**; seed ranges: bootstrap 1001–1128, initialization 2001–2128, minibatches 3001–3128 |

**Primary selection evidence on the server:**

- `/data/repo/MLiP/notebooks/colab_results/dino_sidewalk_data_3_runs_extracted/20261119T230431Z-1cd3eb/selection.json` — `hazard.summary` specifies Pitt-only, learning rate 3e-5,
  weight decay 0.01 and batch size 64. Configurations were compared across three seeds, maximizing
  validation precision@46 and breaking ties with validation hazard balanced accuracy. This
  configuration achieved **71.7391%** mean validation precision@46 and **62.9266%** mean hazard BA.
- `/data/repo/MLiP/notebooks/colab_results/dino_sidewalk_data_3_runs_extracted/20261119T230431Z-1cd3eb/config.json` — records the 5,000-update Pitt training plan, Adam optimizer,
  pinned DINO backbone and preprocessing.
- `/data/repo/MLiP/notebooks/dino_sidewalk_data_3.ipynb` — source experiment implementation.

The earlier decision threshold was 0.5; **no threshold is applied here**, because this notebook
exports probabilities only. The historical selection scores above are provenance, not results
computed for this ensemble. No new sweep, early stopping or member selection is performed.
The new heads use the selected hyperparameters with bootstrap samples, rather than reusing
the original deployment weights. Copies of the selection/configuration records and their original
file hashes are embedded and exported under `configuration_source/`.

Bootstrap sampling is with replacement within strata of exact-pixel duplicate groups, preserving
1,276 draws and the binary class counts for every head. Each group is defined by
`output_rgb_pixel_sha256`; this keeps the 40 duplicate pairs together. There are 1,236 groups.
No geographic IDs are available. Bootstrap counts are exported so out-of-bag analysis can be
performed later without mixing in predictions from heads that trained on the same image.


In [ ]:
%pip install -q "transformers==5.22.1" "gdown==6.0.0" pandas huggingface_hub pillow
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch, transformers
assert torch.cuda.is_available(), 'Select a Colab T4 GPU runtime, then rerun.'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True)
print('GPU:',torch.cuda.get_device_name(0),'| torch:',torch.__version__,'| transformers:',transformers.__version__)


## Input, output and training settings
`DRIVE_FILE_URL` is the file link for `sidewalk-data-3.tar`, not a folder. Anonymous downloads
require that this one file allows Anyone with the link / Viewer. To keep it private, upload it
through VS Code's Upload to Colab action and set `DATA_SOURCE='local_archive'` plus its runtime
path in `LOCAL_ARCHIVE`. Existing archive SHA-256 and image hash checks remain enforced.

Training uses only `pitt-train`. Raw predictions are exported for all three Pittsburgh splits,
including test, without computing evaluation metrics. `RESUME_RUN_DIR` reuses a run still on the
runtime; `RESUME_ARCHIVE` restores a downloaded ZIP uploaded into a new runtime.


In [ ]:
from pathlib import Path
DATA_SOURCE = 'drive_link'
DRIVE_FILE_URL = 'https://drive.google.com/file/d/1w0fxV5KfTeYWMiPPOnKcnN9NYE56SMzD/view?usp=drive_link'
LOCAL_ARCHIVE = Path('/content/sidewalk-data-3.tar')
HF_DATA_REPO = ''
HF_DATA_ARCHIVE = 'sidewalk-data-3.tar'
HF_DATA_REVISION = None
DATA_ROOT = Path('/content/sidewalk-data-3')
RUNS_ROOT = Path('/content/dino_hazard_bootstrap_runs')
DOWNLOAD_DIR = Path('/content/downloads')
RESUME_RUN_DIR = None
RESUME_ARCHIVE = None
CACHE_BATCH_SIZE = 64
PREDICTION_SPLITS = ['pitt-train', 'pitt-val', 'pitt-test']
SETTINGS = dict(n_members=128, lr=3e-5, weight_decay=0.01, batch_size=64,
    updates=5000, log_every=1000, bootstrap_seed_base=1000, init_seed_base=2000, batch_seed_base=3000)
BACKBONE = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
BACKBONE_REVISION = '5931719e67bbdb9737e363e781fb0c67687896bc'
EXPECTED_ARCHIVE_SHA256 = 'd6c88a2b770baefd9ee34b7c846fe8be573258ba6d6ec7cac27e90e2d9eabbbd'
EXPECTED_DATASET_HASHES = {'other-cities-train.csv': 'dc88d9a1fd7683af2f6d551922904035b871577a0b9da79dcc797b79fd5719d5', 'pitt-train.csv': '6fdc86fffcf59152187011a2dd55d5594e63f17ac7abb01529fdf4b79dc63082', 'pitt-val.csv': '32bbce7479f2f366aed2f008be127d7ba4c30fd35aa91feb5e5f704f090319bc', 'pitt-test.csv': 'bff64ae48ef512c0e722cf96cd77f06090411386ef0217242d39740d7600388a', 'label_schema.json': 'b429abdd786bc6c6f191b5fec0d0104171f2722f03ce819ea71f53bdeab911db', 'COMPLETE.json': '40a4b52bf4b07084e5fcb04d0e8f3b712cc7cef282861603a0d0ee58c753b827'}


## Self-contained implementation
This collapsed cell embeds the training, raw inference and ZIP/download helpers. No companion
file needs to be copied into Colab. The executed source is included in the export.


In [ ]:
CORE_SOURCE = '"""Train 128 bootstrap hazard heads and export raw per-member probabilities."""\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib, io, json, os, shutil, tempfile, time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom PIL import Image\nMEAN = [0.485, 0.456, 0.406]\nSTD = [0.229, 0.224, 0.225]\n\n\ndef plain(x):\n    if isinstance(x, Path): return str(x)\n    if isinstance(x, np.generic): return x.item()\n    if isinstance(x, np.ndarray): return x.tolist()\n    raise TypeError(type(x).__name__)\n\ndef canonical(x): return json.dumps(x, sort_keys=True, separators=(\',\', \':\'), default=plain)\n\ndef fingerprint(x): return hashlib.sha256(canonical(x).encode()).hexdigest()\n\ndef file_sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024*1024), b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef atomic_bytes(path, payload):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(\'.\' + path.name + \'.tmp\')\n    tmp.write_bytes(payload); os.replace(tmp, path)\n\ndef save_json(path, value): atomic_bytes(path, (json.dumps(value, indent=2, default=plain, allow_nan=False)+\'\\n\').encode())\n\ndef save_csv(path, frame): atomic_bytes(path, frame.to_csv(index=False).encode())\n\ndef save_torch(path, value):\n    with tempfile.TemporaryDirectory() as temp:\n        local = Path(temp)/\'model.pt\'; torch.save(value, local)\n        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n        tmp = path.with_name(\'.\'+path.name+\'.tmp\'); shutil.copyfile(local, tmp); os.replace(tmp, path)\n\ndef save_npz(path, **arrays):\n    buf = io.BytesIO(); np.savez_compressed(buf, **arrays); atomic_bytes(path, buf.getvalue())\n\ndef batch_indices(n, steps, batch_size, generator, device):\n    assert n>=batch_size\n    order=torch.randperm(n,generator=generator); offset=0\n    for _ in range(steps):\n        if offset+batch_size>n: order=torch.randperm(n,generator=generator);offset=0\n        yield order[offset:offset+batch_size].to(device)\n        offset+=batch_size\n\ndef cache_features(encoder, df, split, root, run_dir, binding, batch_size, device):\n    cache=Path(run_dir)/\'features\'/(split+\'.pt\')\n    cache_binding={**binding,\'split\':split,\'sample_ids_sha256\':fingerprint(df.sample_id.tolist())}\n    if cache.exists():\n        cached=torch.load(cache,map_location=\'cpu\',weights_only=True)\n        assert cached[\'binding\']==cache_binding,\'Feature cache identity mismatch; start a new run.\'\n        x=cached[\'features\']; assert x.shape==(len(df),encoder.config.hidden_size) and torch.isfinite(x).all()\n        print(\'Reused cached features:\',split)\n        return x.to(device).clone()\n    root=Path(root);mean=torch.tensor(MEAN,device=device).view(1,3,1,1);std=torch.tensor(STD,device=device).view(1,3,1,1)\n    def read(row):\n        payload=(root/row.image_path).read_bytes()\n        assert hashlib.sha256(payload).hexdigest()==row.output_file_sha256,f\'Image checksum mismatch: {row.image_path}\'\n        with Image.open(io.BytesIO(payload)) as im:\n            im.load();assert im.size==(224,224) and im.mode==\'RGB\' and im.format==\'PNG\'\n            return np.array(im,copy=True)\n    pieces=[];start=time.perf_counter()\n    with ThreadPoolExecutor(max_workers=8) as pool,torch.inference_mode():\n        for begin in range(0,len(df),batch_size):\n            pixels=np.stack(list(pool.map(read,df.iloc[begin:begin+batch_size].itertuples(index=False))))\n            x=torch.from_numpy(pixels).permute(0,3,1,2).to(device).float()/255\n            x=(x-mean)/std\n            with torch.autocast(device_type=\'cuda\',dtype=torch.float16,enabled=str(device).startswith(\'cuda\')):\n                feats=encoder(pixel_values=x).last_hidden_state[:,0].float()\n            assert torch.isfinite(feats).all(),\'Nonfinite backbone features; stop rather than saving a bad cache.\'\n            pieces.append(feats.cpu())\n            if begin%(batch_size*20)==0:print(f\'{split}: {min(begin+batch_size,len(df))}/{len(df)} embedded\',flush=True)\n    # clone outside inference_mode so head autograd can save its inputs.\n    output=torch.cat(pieces).clone()\n    save_torch(cache,dict(binding=cache_binding,features=output))\n    print(f\'Cached {split} in {time.perf_counter()-start:.1f}s\')\n    return output.to(device).clone()\ndef load_hazard_data(root, expected_hashes):\n    root = Path(root)\n    for filename, expected in expected_hashes.items():\n        if file_sha(root / filename) != expected:\n            raise ValueError(f\'Dataset identity mismatch: {filename}\')\n    schema = json.loads((root / \'label_schema.json\').read_text())\n    assert schema[\'binary_labels\'] == {\'non_hazard\': 0, \'hazard\': 1}\n    data = {}\n    for split in [\'pitt-train\', \'pitt-val\', \'pitt-test\']:\n        df = pd.read_csv(root / (split + \'.csv\'), keep_default_na=False, low_memory=False)\n        assert df.sample_id.is_unique and df.image_path.is_unique\n        assert set(df.split) == {split} and set(df.hazard_label) == {0, 1}\n        assert np.array_equal(df[\'class\'], np.where(df.hazard_label, \'hazard\', \'non_hazard\'))\n        assert df.output_rgb_pixel_sha256.str.fullmatch(\'[0-9a-f]{64}\').all()\n        for relative in df.image_path:\n            path = root / relative\n            assert path.resolve().is_relative_to(root.resolve()) and path.is_file()\n        data[split] = df\n    assert sum(map(len, data.values())) == len(set().union(*(set(df.sample_id) for df in data.values())))\n    return data, schema\n\n\ndef bootstrap_sample(df, seed):\n    """Resample exact-pixel clusters within their binary-label composition strata."""\n    assert set(df.split) == {\'pitt-train\'}, \'Bootstrap only the training split.\'\n    assert df.sample_id.is_unique and set(df.hazard_label) == {0, 1}\n    labels = df.hazard_label.to_numpy(dtype=np.int64)\n    clusters = {}\n    for i, key in enumerate(df.output_rgb_pixel_sha256.astype(str)):\n        assert key\n        clusters.setdefault(key, []).append(i)\n    strata = {}\n    for key, indices in sorted(clusters.items()):\n        composition = (int((labels[indices] == 0).sum()), int((labels[indices] == 1).sum()))\n        strata.setdefault(composition, []).append(key)\n    rng = np.random.default_rng(seed)\n    draws = []\n    for composition, keys in sorted(strata.items()):\n        for key in rng.choice(keys, size=len(keys), replace=True):\n            draws.extend(clusters[key])\n    draws = rng.permutation(np.asarray(draws, dtype=np.int64))\n    counts = np.bincount(draws, minlength=len(df))\n    assert len(draws) == len(df)\n    assert np.array_equal(np.bincount(labels[draws], minlength=2), np.bincount(labels, minlength=2))\n    for indices in clusters.values():\n        assert len(set(counts[indices])) == 1\n    return draws, counts\n\n\ndef bootstrap_frames(df, draws, counts):\n    cols = [\'sample_id\', \'image_path\', \'hazard_label\', \'output_rgb_pixel_sha256\']\n    selected = df.iloc[draws][cols].reset_index(drop=True)\n    selected.insert(0, \'draw_position\', np.arange(len(draws)))\n    selected.insert(1, \'original_row_index\', draws)\n    membership = df[cols].copy()\n    membership.insert(0, \'original_row_index\', np.arange(len(df)))\n    membership[\'in_bag_count\'] = counts\n    membership[\'out_of_bag\'] = counts == 0\n    return selected, membership\n\n\ndef member_ids(entries):\n    ids = np.asarray([entry[\'cfg\'][\'member\'] for entry in entries], dtype=np.int64)\n    assert np.array_equal(ids, np.arange(1, len(entries)+1)), \'Members must be ordered and contiguous.\'\n    return ids\n\n\ndef probability_columns(ids):\n    return [f\'p_hazard_head_{int(member):03d}\' for member in ids]\n\n\n@torch.inference_mode()\ndef predict_member(state, features):\n    head = nn.Linear(features.shape[1], 1).to(features.device)\n    head.load_state_dict(state)\n    return head.eval()(features).sigmoid().float().cpu().numpy()[:, 0]\n\n\n@torch.inference_mode()\ndef predict_ensemble(entries, features):\n    member_ids(entries)\n    weight = torch.cat([entry[\'state_dict\'][\'weight\'] for entry in entries]).to(features.device)\n    bias = torch.cat([entry[\'state_dict\'][\'bias\'] for entry in entries]).to(features.device)\n    p = (features @ weight.T + bias).sigmoid().float().cpu().numpy()\n    assert p.shape == (len(features), len(entries)) and np.isfinite(p).all()\n    assert ((p >= 0) & (p <= 1)).all()\n    return p\n\n\ndef train_member(cfg, train_features, train_df, draws, settings):\n    """Only training features/labels enter optimization; retain the final update."""\n    assert set(train_df.split) == {\'pitt-train\'}\n    device = train_features.device\n    torch.manual_seed(cfg[\'init_seed\'])\n    generator = torch.Generator().manual_seed(cfg[\'batch_seed\'])\n    head = nn.Linear(train_features.shape[1], 1).to(device)\n    optimizer = torch.optim.Adam(head.parameters(), lr=settings[\'lr\'], weight_decay=settings[\'weight_decay\'])\n    x = train_features[torch.as_tensor(draws, device=device)]\n    y = torch.as_tensor(train_df.hazard_label.to_numpy()[draws], device=device, dtype=torch.float32)\n    history, window = [], []\n    for step, idx in enumerate(batch_indices(len(x), settings[\'updates\'], settings[\'batch_size\'], generator, device), 1):\n        loss = nn.functional.binary_cross_entropy_with_logits(head(x[idx])[:, 0], y[idx])\n        optimizer.zero_grad(set_to_none=True)\n        loss.backward()\n        optimizer.step()\n        window.append(loss.detach())\n        if step % settings[\'log_every\'] == 0 or step == settings[\'updates\']:\n            mean_loss = float(torch.stack(window).mean())\n            assert np.isfinite(mean_loss), \'Nonfinite training loss.\'\n            history.append(dict(member=cfg[\'member\'], step=step, train_bce=mean_loss))\n            window = []\n    state = {name: tensor.detach().cpu().clone() for name, tensor in head.state_dict().items()}\n    assert all(torch.isfinite(tensor).all() for tensor in state.values())\n    return state, pd.DataFrame(history)\n\n\ndef run_ensemble(settings, binding, train_features, train_df, run_dir):\n    assert settings[\'n_members\'] == 128, \'This experiment trains exactly 128 heads.\'\n    assert set(train_df.split) == {\'pitt-train\'} and len(train_features) == len(train_df)\n    entries, histories, summaries = [], [], []\n    for member in range(1, settings[\'n_members\']+1):\n        cfg = dict(member=member, bootstrap_seed=settings[\'bootstrap_seed_base\']+member,\n                   init_seed=settings[\'init_seed_base\']+member, batch_seed=settings[\'batch_seed_base\']+member)\n        draws, counts = bootstrap_sample(train_df, cfg[\'bootstrap_seed\'])\n        folder = Path(run_dir)/\'members\'/f\'head_{member:03d}\'\n        folder.mkdir(parents=True, exist_ok=True)\n        done = folder/\'complete.json\'\n        identity = dict(cfg=cfg, settings=settings, binding=binding,\n                        train_sample_ids_sha256=fingerprint(train_df.sample_id.tolist()),\n                        bootstrap_draws_sha256=fingerprint(draws))\n        if done.exists():\n            completion = json.loads(done.read_text())\n            assert canonical(completion[\'identity\']) == canonical(identity), \'Resume identity mismatch.\'\n            for name, expected in completion[\'file_hashes\'].items():\n                assert file_sha(folder/name) == expected, f\'Damaged member artifact: {name}\'\n            payload = torch.load(folder/\'head.pt\', map_location=\'cpu\', weights_only=True)\n            assert canonical(payload[\'identity\']) == canonical(identity)\n            state, history = payload[\'state_dict\'], pd.read_csv(folder/\'history.csv\')\n            action = \'reused\'\n        else:\n            selected, membership = bootstrap_frames(train_df, draws, counts)\n            save_csv(folder/\'bootstrap_draws.csv\', selected)\n            save_csv(folder/\'bootstrap_counts.csv\', membership)\n            start = time.perf_counter()\n            state, history = train_member(cfg, train_features, train_df, draws, settings)\n            save_torch(folder/\'head.pt\', dict(format_version=2, task=\'hazard\', identity=identity,\n                       feature_dim=train_features.shape[1], output_labels=[\'hazard\'], state_dict=state))\n            reloaded = torch.load(folder/\'head.pt\', map_location=\'cpu\', weights_only=True)\n            assert all(torch.equal(state[k], reloaded[\'state_dict\'][k]) for k in state)\n            save_csv(folder/\'history.csv\', history)\n            save_json(folder/\'timing.json\', dict(training_and_save_seconds=time.perf_counter()-start))\n            files = [\'head.pt\', \'bootstrap_draws.csv\', \'bootstrap_counts.csv\', \'history.csv\', \'timing.json\']\n            save_json(done, dict(identity=identity, file_hashes={name:file_sha(folder/name) for name in files}))\n            action = \'trained\'\n        entries.append(dict(cfg=cfg, state_dict=state, counts=counts))\n        histories.append(history)\n        summaries.append(dict(**cfg, bootstrap_rows=len(draws), unique_rows=int((counts>0).sum()),\n                              out_of_bag_rows=int((counts==0).sum()), hazards=int(train_df.hazard_label.to_numpy()[draws].sum()),\n                              bootstrap_draws_sha256=fingerprint(draws)))\n        save_json(Path(run_dir)/\'status.json\', dict(status=\'training\', members_completed=member, members_total=settings[\'n_members\']))\n        print(f"[{member}/{settings[\'n_members\']}] {action}; {settings[\'updates\']} updates; {int((counts>0).sum())} unique training records", flush=True)\n    assert len({s[\'bootstrap_draws_sha256\'] for s in summaries}) == settings[\'n_members\']\n    history = pd.concat(histories, ignore_index=True)\n    save_csv(Path(run_dir)/\'training_history.csv\', history)\n    save_csv(Path(run_dir)/\'bootstrap_members.csv\', pd.DataFrame(summaries))\n    save_npz(Path(run_dir)/\'pitt-train_bootstrap_counts.npz\',\n             counts=np.column_stack([entry[\'counts\'] for entry in entries]),\n             sample_ids=train_df.sample_id.to_numpy(dtype=str), member_ids=member_ids(entries))\n    return entries\n\n\ndef export_predictions(entries, features, df, run_dir, split):\n    assert set(df.split) == {split} and len(df) == len(features)\n    ids = member_ids(entries)\n    p = predict_ensemble(entries, features)\n    metadata_columns = [\'sample_id\', \'image_path\', \'split\', \'hazard_label\', \'original_class\',\n                        \'city\', \'is_manual_collection\', \'output_file_sha256\', \'output_rgb_pixel_sha256\']\n    frame = pd.concat([df[[c for c in metadata_columns if c in df]].reset_index(drop=True),\n                       pd.DataFrame(p, columns=probability_columns(ids))], axis=1)\n    save_csv(Path(run_dir)/(split+\'_predictions.csv\'), frame)\n    save_npz(Path(run_dir)/(split+\'_probabilities.npz\'), probabilities=p,\n             sample_ids=df.sample_id.to_numpy(dtype=str), member_ids=ids)\n    return frame\n\n\ndef export_ensemble(entries, binding, check_features, run_dir):\n    root = Path(run_dir)/\'inference\'\n    root.mkdir(exist_ok=True)\n    ids = member_ids(entries)\n    payload = dict(format_version=2, task=\'hazard\', n_members=len(entries), feature_dim=check_features.shape[1],\n                   binding=binding, member_configs=[e[\'cfg\'] for e in entries], member_ids=ids.tolist(),\n                   probability_columns=probability_columns(ids),\n                   weight=torch.cat([e[\'state_dict\'][\'weight\'] for e in entries]),\n                   bias=torch.cat([e[\'state_dict\'][\'bias\'] for e in entries]))\n    if (root/\'ensemble.pt\').exists():\n        previous = torch.load(root/\'ensemble.pt\', map_location=\'cpu\', weights_only=True)\n        for key in [\'format_version\', \'task\', \'n_members\', \'feature_dim\', \'binding\', \'member_configs\', \'member_ids\', \'probability_columns\']:\n            assert canonical(previous[key]) == canonical(payload[key]), \'Export identity changed.\'\n        assert torch.equal(previous[\'weight\'], payload[\'weight\']) and torch.equal(previous[\'bias\'], payload[\'bias\'])\n    else:\n        save_torch(root/\'ensemble.pt\', payload)\n    saved = torch.load(root/\'ensemble.pt\', map_location=check_features.device, weights_only=True)\n    with torch.inference_mode():\n        p = (check_features @ saved[\'weight\'].T + saved[\'bias\']).sigmoid().cpu().numpy()\n    expected = np.column_stack([predict_member(e[\'state_dict\'], check_features) for e in entries])\n    error = float(np.max(np.abs(p-expected)))\n    assert error < 2e-6\n    save_json(root/\'checkpoint_reload_verification.json\', dict(passed=True, max_probability_difference=error, n_members=len(entries)))\n    return root\n'
INFERENCE_SOURCE = '"""Offline export of raw hazard probabilities from all saved bootstrap heads."""\nfrom pathlib import Path\nimport argparse\nimport csv\nimport json\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageOps\n\n\nclass HazardEnsemble:\n    def __init__(self, bundle, device=None):\n        from transformers import AutoModel\n        self.bundle = Path(bundle)\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.preprocessing = json.loads((self.bundle/\'preprocessing.json\').read_text())\n        saved = torch.load(self.bundle/\'ensemble.pt\', map_location=self.device, weights_only=True)\n        assert saved[\'format_version\'] == 2 and saved[\'task\'] == \'hazard\'\n        self.n_members = saved[\'n_members\']\n        self.columns = saved[\'probability_columns\']\n        self.weight, self.bias = saved[\'weight\'], saved[\'bias\']\n        assert self.weight.shape[0] == self.bias.shape[0] == len(self.columns) == self.n_members\n        self.encoder = AutoModel.from_pretrained(self.bundle/\'backbone\', local_files_only=True).to(self.device).eval().requires_grad_(False)\n        self.mean = torch.tensor(self.preprocessing[\'mean\'], device=self.device).view(1,3,1,1)\n        self.std = torch.tensor(self.preprocessing[\'std\'], device=self.device).view(1,3,1,1)\n        if str(self.device).startswith(\'cuda\'):\n            torch.backends.cuda.matmul.allow_tf32 = False\n            torch.backends.cudnn.allow_tf32 = False\n\n    @torch.inference_mode()\n    def predict(self, paths, batch_size=64):\n        paths = [Path(p) for p in paths]\n        assert len(paths) and len({str(p) for p in paths}) == len(paths)\n        assert batch_size > 0\n        rows = []\n        for start in range(0, len(paths), batch_size):\n            chosen, pixels = paths[start:start+batch_size], []\n            for path in chosen:\n                with Image.open(path) as image:\n                    image = ImageOps.exif_transpose(image).convert(\'RGB\').resize(tuple(self.preprocessing[\'size\']), Image.Resampling.BICUBIC)\n                    pixels.append(np.array(image, copy=True))\n            x = torch.from_numpy(np.stack(pixels)).permute(0,3,1,2).to(self.device).float()/255\n            with torch.autocast(device_type=\'cuda\', dtype=torch.float16, enabled=str(self.device).startswith(\'cuda\')):\n                features = self.encoder(pixel_values=(x-self.mean)/self.std).last_hidden_state[:,0].float()\n            p = (features @ self.weight.T + self.bias).sigmoid().cpu().numpy()\n            assert p.shape == (len(chosen), self.n_members) and np.isfinite(p).all()\n            for i, path in enumerate(chosen):\n                row = dict(image_path=str(path))\n                row.update({column:float(p[i,j]) for j,column in enumerate(self.columns)})\n                rows.append(row)\n        return rows\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--bundle\', type=Path, required=True)\n    parser.add_argument(\'--images\', nargs=\'+\', required=True)\n    parser.add_argument(\'--output\', type=Path, default=Path(\'hazard_predictions.csv\'))\n    parser.add_argument(\'--device\', default=None)\n    args = parser.parse_args()\n    predictor = HazardEnsemble(args.bundle, args.device)\n    rows = predictor.predict(args.images)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    with args.output.open(\'w\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    print(f\'Saved {len(rows)} images with {predictor.n_members} raw hazard probabilities: {args.output}\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
IO_SOURCE = '"""Dataset download and result ZIPs without mounting or authenticating Google Drive."""\nfrom pathlib import Path, PurePosixPath\nfrom urllib.parse import urlsplit\nimport hashlib\nimport json\nimport os\nimport stat\nimport tempfile\nimport zipfile\n\n\ndef file_sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef download_drive_archive(url, destination, expected_sha256):\n    """Download a publicly readable file; never load browser/account cookies."""\n    destination = Path(destination)\n    if destination.exists():\n        if file_sha(destination) != expected_sha256:\n            raise ValueError(f\'{destination} exists but is not the expected archive; move it aside before downloading.\')\n        return destination\n    parsed = urlsplit(str(url).strip())\n    if parsed.scheme != \'https\' or parsed.hostname not in {\'drive.google.com\', \'docs.google.com\'}:\n        raise ValueError(\'Paste the HTTPS Google Drive share link for sidewalk-data-3.tar.\')\n    if \'/folders/\' in parsed.path:\n        raise ValueError(\'Supply the archive file link, not a folder link.\')\n    import gdown\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_name(destination.name+\'.download\')\n    try:\n        result = gdown.download(url=str(url).strip(), output=str(partial), use_cookies=False, quiet=False, resume=True)\n    except Exception:\n        raise RuntimeError(\'Could not download the archive anonymously. The file must allow Anyone with the link / Viewer. \'\n                           \'Alternatively upload it to the Colab runtime and select DATA_SOURCE="local_archive". \'\n                           \'No Google sign-in or browser cookies are used.\') from None\n    if result is None or not partial.is_file():\n        raise RuntimeError(\'No archive was downloaded. Check the file link, or use DATA_SOURCE="local_archive".\')\n    if file_sha(partial) != expected_sha256:\n        partial.unlink(missing_ok=True)\n        raise ValueError(\'Downloaded file failed the pinned dataset SHA-256 check; it is not the expected dataset-3 archive.\')\n    os.replace(partial, destination)\n    return destination\n\n\ndef make_results_zip(run_dir, destination_dir):\n    """Archive only the run directory, including weights, features and provenance."""\n    root = Path(run_dir).resolve()\n    destination_dir = Path(destination_dir).resolve()\n    assert (root/\'config.json\').is_file(), \'Expected a run directory containing config.json.\'\n    assert not destination_dir.is_relative_to(root), \'Put downloads outside the run directory.\'\n    paths = sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.name.endswith(\'.tmp\'))\n    assert all(not p.is_symlink() and p.resolve().is_relative_to(root) for p in paths)\n    inventory = {str(p.relative_to(root)):dict(bytes=p.stat().st_size, sha256=file_sha(p))\n                 for p in paths if p.name != \'archive_manifest.json\'}\n    (root/\'archive_manifest.json\').write_text(json.dumps(inventory, indent=2)+\'\\n\')\n    paths = sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.name.endswith(\'.tmp\'))\n    destination_dir.mkdir(parents=True, exist_ok=True)\n    output = destination_dir/(root.name+\'.zip\')\n    pending = output.with_name(output.name+\'.tmp\')\n    try:\n        with zipfile.ZipFile(pending, \'w\', compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:\n            for path in paths:\n                assert not path.is_symlink() and path.resolve().is_relative_to(root)\n                archive.write(path, arcname=str(Path(root.name)/path.relative_to(root)))\n        with zipfile.ZipFile(pending) as archive:\n            assert archive.testzip() is None, \'ZIP integrity check failed.\'\n        os.replace(pending, output)\n    finally:\n        pending.unlink(missing_ok=True)\n    checksum = file_sha(output)\n    output.with_suffix(\'.zip.sha256\').write_text(f\'{checksum}  {output.name}\\n\')\n    return output\n\n\ndef restore_results_zip(archive_path, runs_root):\n    """Restore a previously downloaded run ZIP into a fresh runtime."""\n    runs_root = Path(runs_root)\n    runs_root.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as archive:\n        names = []\n        for info in archive.infolist():\n            p = PurePosixPath(info.filename)\n            if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in info.filename or not p.parts:\n                raise ValueError(\'Unsafe path in results archive.\')\n            if stat.S_ISLNK(info.external_attr >> 16):\n                raise ValueError(\'Results archive must not contain symbolic links.\')\n            names.append(p)\n        roots = {p.parts[0] for p in names}\n        if len(roots) != 1:\n            raise ValueError(\'Results archive must contain exactly one run directory.\')\n        run_name = next(iter(roots))\n        if f\'{run_name}/config.json\' not in archive.namelist() or f\'{run_name}/archive_manifest.json\' not in archive.namelist():\n            raise ValueError(\'Results ZIP lacks its configuration or integrity manifest.\')\n        target = runs_root/run_name\n        if target.exists():\n            raise FileExistsError(f\'{target} already exists; set RESUME_RUN_DIR to it instead.\')\n        with tempfile.TemporaryDirectory(dir=runs_root, prefix=\'.restore-\') as temporary:\n            archive.extractall(temporary)\n            candidate = Path(temporary)/run_name\n            inventory = json.loads((candidate/\'archive_manifest.json\').read_text())\n            for relative, expected in inventory.items():\n                path = candidate/relative\n                if not path.resolve().is_relative_to(candidate.resolve()) or not path.is_file():\n                    raise ValueError(\'Invalid inventory path in results ZIP.\')\n                if path.stat().st_size != expected[\'bytes\'] or file_sha(path) != expected[\'sha256\']:\n                    raise ValueError(f\'Result failed integrity verification: {relative}\')\n            candidate.rename(target)\n    return target\n'
SOURCE_RECORDS = {'selection.json': {'hazard': {'summary': {'config_id': '68caf0783493ae93', 'plan': 'pitt', 'lr': 3e-05, 'weight_decay': 0.01, 'batch_size': 64, 'hazard_threshold': 0.5, 'group_thresholds': '[0.5, 0.3, 0.5, 0.3]', 'topk_hits_sum': 99, 'precision_at_k_mean': 0.717391304347826, 'precision_at_k_std': 0.021739130434782594, 'hazard_ba_mean': 0.6292663364000943, 'hazard_ba_std': 0.018758467874733075, 'group_macro_ba_mean': 0.6484636286300097, 'group_macro_ba_std': 0.005524348451560382}, 'trial_ids': ['200e2f618e2a0764', 'c9485a8e42365754', '36e10d55a0a34890'], 'hazard_deployment_trial': 'c9485a8e42365754', 'group_deployment_trial': 'c9485a8e42365754'}, 'tag_groups': {'summary': {'config_id': '22f17857d25b5485', 'plan': 'other_then_pitt', 'lr': 0.0001, 'weight_decay': 0.01, 'batch_size': 64, 'hazard_threshold': 0.5, 'group_thresholds': '[0.3, 0.3, 0.3, 0.3]', 'topk_hits_sum': 89, 'precision_at_k_mean': 0.644927536231884, 'precision_at_k_std': 0.012551092808470117, 'hazard_ba_mean': 0.6320547298891248, 'hazard_ba_std': 0.003655021798486927, 'group_macro_ba_mean': 0.678033546967526, 'group_macro_ba_std': 0.001894866340918123}, 'trial_ids': ['60073cd9ac62e97f', '20281c6fa716c544', '175b6f87d7439c45'], 'hazard_deployment_trial': '60073cd9ac62e97f', 'group_deployment_trial': '175b6f87d7439c45'}, 'k': 46, 'group_order': ['surface_and_level_problems', 'restricted_passage', 'missing_tactile_warning', 'ramp_points_into_traffic'], 'validation_split': 'pitt-val', 'test_used_for_selection': False, 'hazard_rule': 'Maximum mean validation precision@46 (integer summed top-k hits); tie-break maximum mean hazard BA at selected threshold.', 'group_rule': 'Maximum mean macro group BA on ALL true validation hazards, each group threshold selected by its mean BA.', 'seed_rule': 'Configuration selected across three seeds. Single deployment replica selected within winner by same validation objective; smaller seed breaks ties. All selected replicas also saved.', 'threshold_grid': [0.3, 0.5, 0.7], 'checkpoint_rule': 'Final update only; no early-stopping/checkpoint search.'}, 'config.json': {'version': 'dino-sidewalk-data-3-v1', 'settings': {'train_plans': {'pitt': [['pitt-train', 5000]], 'other_then_pitt': [['other-cities-train', 5000], ['pitt-train', 5000]]}, 'learning_rates': [1e-05, 3e-05, 0.0001, 0.0003, 0.001, 0.003, 0.01], 'weight_decays': [0.0, 0.01], 'seeds': [0, 1, 2], 'batch_size': 64, 'eval_every': 250, 'thresholds': [0.3, 0.5, 0.7], 'k': 46}, 'backbone': 'facebook/dinov3-vitb16-pretrain-lvd1689m', 'backbone_revision': '5931719e67bbdb9737e363e781fb0c67687896bc', 'pooling': 'cls', 'backbone_frozen': True, 'expected_dataset_hashes': {'other-cities-train.csv': 'dc88d9a1fd7683af2f6d551922904035b871577a0b9da79dcc797b79fd5719d5', 'pitt-train.csv': '6fdc86fffcf59152187011a2dd55d5594e63f17ac7abb01529fdf4b79dc63082', 'pitt-val.csv': '32bbce7479f2f366aed2f008be127d7ba4c30fd35aa91feb5e5f704f090319bc', 'pitt-test.csv': 'bff64ae48ef512c0e722cf96cd77f06090411386ef0217242d39740d7600388a', 'label_schema.json': 'b429abdd786bc6c6f191b5fec0d0104171f2722f03ce819ea71f53bdeab911db', 'COMPLETE.json': '40a4b52bf4b07084e5fcb04d0e8f3b712cc7cef282861603a0d0ee58c753b827'}, 'group_order': ['surface_and_level_problems', 'restricted_passage', 'missing_tactile_warning', 'ramp_points_into_traffic'], 'core_code_sha256': 'a03d5dd148b5915a774c03f56ccdb276a55182ed4c20fbc88fd4e95f24857ebf', 'inference_code_sha256': '69f53bb9c02c09c6f1486fa722890f333c72e730b98b108276466e4a2af4bb79', 'feature_precision': 'float16', 'head_precision': 'float32', 'preprocessing': {'size': [224, 224], 'resize': 'full_image_bicubic', 'mode': 'RGB', 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]}, 'checkpoint_selection': 'final_step_only', 'optimizer': 'Adam', 'loss': 'unweighted hazard BCE + mean four-group BCE masked to true hazards'}}
CONFIGURATION_SOURCE = {'source_notebook': '/data/repo/MLiP/notebooks/dino_sidewalk_data_3.ipynb', 'source_results_directory': '/data/repo/MLiP/notebooks/colab_results/dino_sidewalk_data_3_runs_extracted/20261119T230431Z-1cd3eb', 'configuration_id': '68caf0783493ae93', 'selection_sha256': 'f157068164a0fa73ba265da500a51f7f66757a59fd4751fb1ee8f0968859f258', 'config_sha256': 'c7d49f9598b2ddd429469fb268e83b648ab89ade4caed3e4f22d7dd1535cfc64', 'selected_configuration': {'plan': 'pitt', 'lr': 3e-05, 'weight_decay': 0.01, 'batch_size': 64, 'hazard_threshold': 0.5, 'precision_at_k_mean': 0.717391304347826, 'hazard_ba_mean': 0.6292663364000943}, 'original_selection_rule': 'Maximum mean validation precision@46 (integer summed top-k hits); tie-break maximum mean hazard BA at selected threshold.', 'original_checkpoint_rule': 'Final update only; no early-stopping/checkpoint search.'}

import importlib.util,sys
def load_module(name,source):
    path=Path('/content')/(name+'.py');path.write_text(source)
    spec=importlib.util.spec_from_file_location(name,path)
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module)
    return module
core=load_module('hazard_bootstrap_core',CORE_SOURCE)
run_io=load_module('hazard_bootstrap_io',IO_SOURCE)


## Create or resume the local run
No Google authentication is performed. The Hugging Face token is read only when needed and is
never included in saved configuration. A partial head restarts; completed heads are verified and
reused. Training settings, code, data and feature environment must match when resuming.


In [ ]:
import json,hashlib,shutil,time,tempfile,tarfile,getpass,importlib.metadata
from datetime import datetime,timezone
from uuid import uuid4
import numpy as np,pandas as pd
assert SETTINGS['n_members']==128
assert PREDICTION_SPLITS==['pitt-train','pitt-val','pitt-test']
HF_TOKEN=None
def get_hf_token():
    global HF_TOKEN
    if HF_TOKEN:return HF_TOKEN
    HF_TOKEN=os.environ.get('HF_TOKEN','').strip()
    if not HF_TOKEN:
        try:
            from google.colab import userdata
            HF_TOKEN=(userdata.get('HF_TOKEN') or '').strip()
        except Exception:HF_TOKEN=''
    if not HF_TOKEN:HF_TOKEN=getpass.getpass('HF read token with DINOv3 access: ').strip()
    if not HF_TOKEN:raise RuntimeError('DINOv3 requires a token with access to its gated model.')
    return HF_TOKEN

run_config=dict(version='hazard-bootstrap-raw-v2',settings=SETTINGS,training_split='pitt-train',
    task='hazard_only',backbone=BACKBONE,backbone_revision=BACKBONE_REVISION,
    expected_dataset_hashes=EXPECTED_DATASET_HASHES,pooling='cls',backbone_frozen=True,
    core_sha256=hashlib.sha256(CORE_SOURCE.encode()).hexdigest(),
    inference_sha256=hashlib.sha256(INFERENCE_SOURCE.encode()).hexdigest(),
    configuration_source=CONFIGURATION_SOURCE,
    optimizer='Adam',optimizer_betas=[0.9,0.999],optimizer_eps=1e-8,
    loss='unweighted hazard binary cross entropy with logits',checkpoint_rule='final_update_only',
    bootstrap='replacement sampling of exact-pixel clusters stratified by binary-label composition',
    export='raw independent probabilities; no aggregation, thresholds, ranking or evaluation',
    prediction_splits=PREDICTION_SPLITS,
    preprocessing=dict(size=[224,224],mode='RGB',resize='full_image_bicubic',mean=core.MEAN,std=core.STD),
    feature_precision='float16',head_precision='float32')
assert not (RESUME_RUN_DIR and RESUME_ARCHIVE),'Set only one resume option.'
if RESUME_ARCHIVE:RESUME_RUN_DIR=run_io.restore_results_zip(RESUME_ARCHIVE,RUNS_ROOT)
RUN_DIR=Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else RUNS_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid4().hex[:6])
if RESUME_RUN_DIR:
    assert (RUN_DIR/'config.json').is_file(),'Resume directory has no config.json.'
    assert core.canonical(json.loads((RUN_DIR/'config.json').read_text()))==core.canonical(run_config),'Training/code/data changed or this is an old 10-head run; use a fresh run.'
else:
    RUN_DIR.mkdir(parents=True,exist_ok=False);core.save_json(RUN_DIR/'config.json',run_config)
environment=dict(python=sys.version,gpu=torch.cuda.get_device_name(0),torch=str(torch.__version__),
    transformers=transformers.__version__,cuda=str(torch.version.cuda),
    packages={p:importlib.metadata.version(p) for p in ['numpy','pandas','huggingface-hub','Pillow','gdown']})
feature_binding=dict(run_config_sha256=core.fingerprint(run_config),backbone_revision=BACKBONE_REVISION,
    gpu=environment['gpu'],torch=environment['torch'],transformers=environment['transformers'],precision='float16',
    torch_tf32=False,cache_batch_size=CACHE_BATCH_SIZE)
if (RUN_DIR/'feature_binding.json').exists():
    assert json.loads((RUN_DIR/'feature_binding.json').read_text())==feature_binding,'Feature environment changed; use a new run.'
else:core.save_json(RUN_DIR/'feature_binding.json',feature_binding)
core.save_json(RUN_DIR/'environment.json',environment)
for name,source in [('hazard_bootstrap_core.py',CORE_SOURCE),('hazard_bootstrap_inference.py',INFERENCE_SOURCE),('hazard_bootstrap_io.py',IO_SOURCE)]:
    (RUN_DIR/name).write_text(source)
core.save_json(RUN_DIR/'storage.json',dict(mode='local_runtime_plus_manual_zip_download',google_authorization_used=False))
for name,record in SOURCE_RECORDS.items():core.save_json(RUN_DIR/'configuration_source'/name,record)
core.save_json(RUN_DIR/'configuration_source/provenance.json',CONFIGURATION_SOURCE)
print('Local run:',RUN_DIR)
print('128 hazard heads, 5,000 updates each; predictions only, no analysis.')


## Load the pinned dataset and retain its metadata
The original manifests are copied to the results for downstream joins and label audits.
The dataset loader verifies split identity and hazard labels. No validation/test score is computed.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
already_extracted = all((DATA_ROOT/name).is_file() for name in EXPECTED_DATASET_HASHES)
if not already_extracted:
    if DATA_ROOT.exists(): raise RuntimeError(f'{DATA_ROOT} is incomplete; choose a fresh DATA_ROOT or remove your incomplete extraction.')
    if DATA_SOURCE == 'drive_link':
        if not LOCAL_ARCHIVE.is_file() and not DRIVE_FILE_URL:
            DRIVE_FILE_URL = input('Paste the file share link for sidewalk-data-3.tar: ').strip()
        local_tar = run_io.download_drive_archive(DRIVE_FILE_URL, LOCAL_ARCHIVE, EXPECTED_ARCHIVE_SHA256)
        download_info = dict(source='anonymous_drive_file_download',google_authorization_used=False)
    elif DATA_SOURCE == 'local_archive':
        local_tar = Path(LOCAL_ARCHIVE)
        if not local_tar.is_file():
            raise FileNotFoundError(f'Upload sidewalk-data-3.tar to the Colab runtime and set LOCAL_ARCHIVE to its path; currently {local_tar}.')
        download_info = dict(source='local_uploaded_archive',google_authorization_used=False)
    elif DATA_SOURCE == 'huggingface':
        assert HF_DATA_REPO, 'Set HF_DATA_REPO to the repository containing sidewalk-data-3.tar.'
        revision=HF_DATA_REVISION or HfApi(token=get_hf_token()).dataset_info(HF_DATA_REPO).sha
        local_tar=Path(hf_hub_download(HF_DATA_REPO,HF_DATA_ARCHIVE,repo_type='dataset',revision=revision,token=get_hf_token()))
        download_info=dict(source='huggingface',repo=HF_DATA_REPO,revision=revision,archive=HF_DATA_ARCHIVE)
    else:
        raise FileNotFoundError('DATA_SOURCE=existing requires the extracted dataset at DATA_ROOT.')
    assert core.file_sha(local_tar)==EXPECTED_ARCHIVE_SHA256, 'Archive is not the prepared dataset-3 snapshot.'
    DATA_ROOT.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(dir=DATA_ROOT.parent,prefix='sidewalk3-extract-') as temp:
        with tarfile.open(local_tar) as archive:
            for member in archive.getmembers():
                path=Path(member.name)
                assert not path.is_absolute() and '..' not in path.parts and path.parts[0]=='sidewalk-data-3'
                assert member.isfile() or member.isdir(), 'Archive contains a non-regular entry.'
            archive.extractall(temp,filter='data')
        Path(temp,'sidewalk-data-3').rename(DATA_ROOT)
    core.save_json(RUN_DIR/'download.json',{**download_info,'sha256':EXPECTED_ARCHIVE_SHA256})

data,schema=core.load_hazard_data(DATA_ROOT,EXPECTED_DATASET_HASHES)
assert {s:len(df) for s,df in data.items()}=={'pitt-train':1276,'pitt-val':382,'pitt-test':380}
snapshot=RUN_DIR/'dataset_metadata';snapshot.mkdir(exist_ok=True)
for name in list(EXPECTED_DATASET_HASHES)+['README.md','LABELING_RULES.md','PREPROCESSING.md','tag_mapping.csv']:
    shutil.copy2(DATA_ROOT/name,snapshot/name)

print('Training rows:',len(data['pitt-train']))
print('Prediction rows:',{s:len(data[s]) for s in PREDICTION_SPLITS})


## Cache the training features and save the frozen backbone
The encoder is frozen and evaluated once per image. Training reuses its cached 768-dimensional
vectors; it does not repeatedly run DINO or update its weights. Only training features are loaded
at this stage. The backbone is saved once for later offline inference.


In [ ]:
from transformers import AutoModel
encoder=AutoModel.from_pretrained(BACKBONE,revision=BACKBONE_REVISION,token=get_hf_token()).to('cuda').eval().requires_grad_(False)
assert not any(p.requires_grad for p in encoder.parameters())
features={}
for split in ['pitt-train']:
    features[split]=core.cache_features(encoder,data[split],split,DATA_ROOT,RUN_DIR,feature_binding,CACHE_BATCH_SIZE,'cuda')

inference_dir=RUN_DIR/'inference';inference_dir.mkdir(exist_ok=True)
backbone_dir=inference_dir/'backbone'
if backbone_dir.exists():
    assert (inference_dir/'backbone_files.json').exists(), 'Incomplete backbone export.'
    for relative,expected in json.loads((inference_dir/'backbone_files.json').read_text()).items():
        assert core.file_sha(backbone_dir/relative)==expected
else:
    pending=inference_dir/'.backbone_pending'
    if pending.exists(): shutil.rmtree(pending)  # only this run's interrupted export
    with tempfile.TemporaryDirectory() as temp:
        encoder.save_pretrained(temp,safe_serialization=True)
        shutil.copytree(temp,pending)
    hashes={str(p.relative_to(pending)):core.file_sha(p) for p in pending.rglob('*') if p.is_file()}
    core.save_json(inference_dir/'backbone_files.json',hashes)
    pending.rename(backbone_dir)
print('Backbone and training features saved on this runtime; included in the ZIP.')


## Train all 128 heads
Every head receives its own bootstrap sample, initialization and minibatch sequence. Only
`pitt-train` features and labels are passed to training. All 128 final checkpoints are saved;
there is no validation pass, early stopping or member selection. Loss logging is a lightweight
optimization record. Completed-head files are written immediately for resumption.


In [ ]:
try:
    entries=core.run_ensemble(SETTINGS,feature_binding,features['pitt-train'],data['pitt-train'],RUN_DIR)
except BaseException as error:
    core.save_json(RUN_DIR/'status.json',dict(status='interrupted',error_type=type(error).__name__))
    try:print('Partial results ZIP:',run_io.make_results_zip(RUN_DIR,DOWNLOAD_DIR))
    except Exception as packaging_error:print('ZIP packaging failed:',type(packaging_error).__name__,'; saved heads remain at',RUN_DIR)
    raise
print('All 128 final heads are saved:',RUN_DIR/'members')


## Save the inference bundle and export raw probabilities
Each split CSV contains identifiers, metadata/ground-truth labels and exactly 128 probability
columns named `p_hazard_head_001` through `p_hazard_head_128`. Rows retain manifest order. The
same columns identify the same heads across train, validation, test and future inference.
NPZ files store the `(number of images, 128)` float32 matrix, `sample_ids` and `member_ids`.

There are no mean/std columns, binary decisions, rankings, metric tables or plots. For later
out-of-bag analysis, `pitt-train_bootstrap_counts.npz` contains an aligned `(1276, 128)` count
matrix: zero means that head did not train on that record's duplicate group. Training predictions
otherwise include heads that trained on each observation. The original label/class metadata
is retained for external analysis, but no subclass predictions are produced.


In [ ]:
inference_dir=core.export_ensemble(entries,feature_binding,features['pitt-train'][:8],RUN_DIR)
(inference_dir/'hazard_bootstrap_inference.py').write_text(INFERENCE_SOURCE)
core.save_json(inference_dir/'preprocessing.json',dict(size=[224,224],mode='RGB',exif_transpose=True,
    resize='full_image_bicubic',mean=core.MEAN,std=core.STD,pooling='cls',feature_precision='float16',
    backbone=BACKBONE,backbone_revision=BACKBONE_REVISION))
core.save_json(inference_dir/'configuration_source.json',CONFIGURATION_SOURCE)
core.save_json(RUN_DIR/'prediction_schema.json',dict(task='hazard',n_members=128,
    columns=core.probability_columns(core.member_ids(entries)),member_ids=core.member_ids(entries),
    row_order='original split manifest order',probability='sigmoid(logit) for each independent head',
    aggregation_applied=False,decision_threshold_applied=False,analysis_performed=False,
    bootstrap_counts_file='pitt-train_bootstrap_counts.npz'))
requirements='\n'.join(['torch=='+str(torch.__version__).split('+')[0],'transformers=='+transformers.__version__,
    'numpy=='+importlib.metadata.version('numpy'),'Pillow=='+importlib.metadata.version('Pillow')])+'\n'
(inference_dir/'requirements.txt').write_text(requirements)
(inference_dir/'README.md').write_text(
    '# 128 bootstrap hazard heads\n\nKeep the entire inference/ directory, including backbone/.\n'
    'Run: python hazard_bootstrap_inference.py --bundle . --images image1.png image2.jpg --output predictions.csv\n'
    'Outputs are raw p_hazard_head_001 ... p_hazard_head_128, with no aggregation, decisions or ranking.\n'
    'No network/HF token is needed for the complete offline bundle. CPU FP32 backbone operations may\n'
    'differ slightly from the T4 FP16 feature extraction used in the experiment.\n')
for split in PREDICTION_SPLITS:
    if split not in features:
        features[split]=core.cache_features(encoder,data[split],split,DATA_ROOT,RUN_DIR,feature_binding,CACHE_BATCH_SIZE,'cuda')
    frame=core.export_predictions(entries,features[split],data[split],RUN_DIR,split)
    print(split,':',len(frame),'images x 128 raw probabilities saved')
core.save_json(RUN_DIR/'status.json',dict(status='complete',members_completed=128,members_total=128,
    prediction_splits=PREDICTION_SPLITS,analysis_performed=False))
print('Inference bundle:',inference_dir)


## Package and download the results
The ZIP contains all 128 heads, the frozen backbone, raw prediction CSVs/NPZs, original manifests,
bootstrap membership, training logs and source/provenance. Dataset images are not included.

**VS Code:** open the Colab sidebar, expand your runtime's files, open `downloads` under `/content`,
right-click the ZIP and select **Download…**. On older extensions, update or enable
`colab.serverMounting` and use **Colab: Mount Server to Workspace…**. This exposes runtime files;
it does not mount Google Drive. Browser Colab users can download through the Files sidebar.

Download before resetting/deleting the runtime. Saving the notebook alone does not save these
artifacts. To resume this implementation later, upload its ZIP and set `RESUME_ARCHIVE`.
All ranking and analysis can then run outside Colab using the exported probability matrices.


In [ ]:
RESULTS_ZIP=run_io.make_results_zip(RUN_DIR,DOWNLOAD_DIR)
print('Download:',RESULTS_ZIP)
print('Size: %.1f MiB' % (RESULTS_ZIP.stat().st_size/1024**2))
print('SHA-256:',run_io.file_sha(RESULTS_ZIP))
print('VS Code: Colab sidebar > runtime files > downloads > right-click ZIP > Download...')
